In [ ]:
import vrplib, re

def load_instance(family, n, k):
    base = f"instances/{family}/{family}-n{n}-k{k}"
    instance = vrplib.read_instance(f"{base}.vrp")
    solution = vrplib.read_solution(f"{base}.sol")
    print(instance["name"], instance["type"], instance["edge_weight_type"], "successfully loaded")
    print("dimension", instance["dimension"])
    print("capacity", instance["capacity"])
    print(instance["comment"])
    return instance, solution

instance, solution = load_instance("A", 32, 5)

k = int(re.search(r"No of trucks: (\d+)", instance["comment"]).group(1))

In [ ]:
def initial_solution(instance):
    n = instance["dimension"]
    return [[c] for c in range(1, n)]

def route_cost(route, edge_weight, depot=0):
    total = 0
    position = depot
    for client in route:
        total += edge_weight[position][client]
        position = client
    total += edge_weight[position][depot]
    return total

def total_cost(routes, edge_weight):
    return sum(route_cost(r, edge_weight) for r in routes)

In [ ]:
print(instance['depot'], instance['node_coord'], instance['demand'], instance['edge_weight'])

solution.keys()

In [ ]:
def vehicle_load(route, demand):
    total = 0
    for client in route:
        total += demand[client]
    return total

def is_possible(routes, instance, k):
    if len(routes) > k:
        return False
    for route in routes:
        if vehicle_load(route, instance["demand"]) > instance["capacity"]:
            return False
    clients = []
    for route in routes:
        clients += route
    if sorted(clients) != list(range(1, instance["dimension"])):
        return False
    return True

In [ ]:
def greedy_initial_solution(instance):
    demand = instance["demand"]
    capacity = instance["capacity"]

    routes = []
    current_route = []
    current_load = 0

    for client in range(1, instance["dimension"]):
        if current_load + demand[client] > capacity:
            routes.append(current_route)   # on ferme le camion
            current_route = []             # nouveau camion vide
            current_load = 0
        current_route.append(client)
        current_load += demand[client]

    routes.append(current_route)           # ne pas oublier le dernier camion
    return routes

In [ ]:
routes = greedy_initial_solution(instance)

for r in routes:
    print(r, "-> charge", vehicle_load(r, instance["demand"]))

print("nombre de routes :", len(routes))
print("faisable ?       :", is_possible(routes, instance, k))
print("coût             :", round(total_cost(routes, instance["edge_weight"]), 2))

In [ ]:
import random
import copy

random.seed(8500081009970950196)

def relocate(routes):
    nouvelles_routes = copy.deepcopy(routes)

    # on choisit un camion au hasard, qui n'est pas vide
    camions_non_vides = []
    for i in range(len(nouvelles_routes)):
        if len(nouvelles_routes[i]) > 0:
            camions_non_vides.append(i)
    camion_depart = random.choice(camions_non_vides)

    # on choisit un client dans ce camion et on l'enlève
    position = random.randrange(len(nouvelles_routes[camion_depart]))
    client = nouvelles_routes[camion_depart].pop(position)

    # on choisit un camion d'arrivée et une place dedans
    camion_arrivee = random.randrange(len(nouvelles_routes))
    nouvelle_place = random.randint(0, len(nouvelles_routes[camion_arrivee]))
    nouvelles_routes[camion_arrivee].insert(nouvelle_place, client)

    return nouvelles_routes

In [ ]:
routes = greedy_initial_solution(instance)
voisin = relocate(routes)

print("avant :", routes)
print("après :", voisin)
print("faisable ?", is_possible(voisin, instance, k))
print("coût avant :", total_cost(routes, instance["edge_weight"]))
print("coût après :", total_cost(voisin, instance["edge_weight"]))

In [ ]:
import matplotlib.pyplot as plt

def hill_climbing(routes, instance, iterations=1000):
    history = []
    best_routes = routes
    for _ in range(iterations):
        voisin = relocate(best_routes)
        if is_possible(voisin, instance, k) and total_cost(voisin, instance["edge_weight"]) <= total_cost(best_routes, instance["edge_weight"]):
            best_routes = voisin
            history.append(total_cost(best_routes, instance["edge_weight"]))
    voisin = best_routes
    return(voisin, history)

voisin, history = hill_climbing(greedy_initial_solution(instance), instance)

fig, ax = plt.subplots()
ax.plot(history)
ax.set(xlabel='itération', ylabel='coût total', title='Convergence')
ax.grid()
fig.savefig("convergence.png")
plt.show()

print("avant :", routes)
print("après :", voisin)
print("faisable ?", is_possible(voisin, instance, k))
print("coût avant :", total_cost(routes, instance["edge_weight"]))
print("coût après :", total_cost(voisin, instance["edge_weight"]))

In [ ]:
from math import exp

def get_proba(delta, t):
    return exp(-(delta/t))

def simulated_annealing(routes, instance, t=70, alpha=0.995, iterations=1000):
    current = copy.deepcopy(routes)
    current_cost = total_cost(current, instance["edge_weight"])
    best = copy.deepcopy(current)
    best_cost = current_cost
    new_cost = current_cost
    hist_current, hist_best = [], []
    
    for _ in range(iterations):
        voisin = relocate(copy.deepcopy(current))
        if is_possible(voisin, instance, k):
            new_cost = total_cost(voisin, instance["edge_weight"])
            delta = new_cost - current_cost
            if delta <= 0 or random.random() < get_proba(delta, t):
                current, current_cost = voisin, new_cost
                if current_cost < best_cost:
                    best, best_cost = copy.deepcopy(current), current_cost
        t *= alpha
        hist_current.append(new_cost)
        hist_best.append(best_cost)

    return best, best_cost, hist_current, hist_best


routes = greedy_initial_solution(instance)
best, best_cost, hist_current, hist_best = simulated_annealing(routes, instance, k)
fig, ax = plt.subplots()
ax.plot(hist_current)
ax.plot(hist_best)
ax.set(xlabel='itération', ylabel='coût total', title='Convergence')
ax.grid()
fig.savefig("convergence.png")
plt.show()

print("avant :", routes)
print("après :", voisin)
print("faisable ?", is_possible(voisin, instance, k))
print("coût avant :", total_cost(routes, instance["edge_weight"]))
print("coût après :", total_cost(voisin, instance["edge_weight"]))